## Training the Model using the MNIST Dataset

#### Loading the Dataset

In [22]:
import torch
import torch.nn as nn
import torchvision
import pandas as pd
import numpy as np

In [2]:
## Importing the Dataset
from torchvision.datasets import MNIST

In [3]:
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

In [4]:
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 342kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.14MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.93MB/s]


#### Implementing the Convolution Neural Networks

In [5]:
model = nn.Sequential(
    nn.Conv2d(
        in_channels = 1,
        out_channels = 16,
        kernel_size = 3,
        stride = 1,
        padding = 0,
    ),
    nn.ReLU(),
    nn.MaxPool2d(
        kernel_size = 2,
        stride = 2
    ),
    nn.Conv2d(
        in_channels= 16,
        out_channels=32,
        kernel_size= 3,
        stride = 1,
        padding = 0,
    ),
    nn.ReLU(),
    nn.MaxPool2d(
        kernel_size = 2,
        stride = 2
    ),
    nn.Flatten(),
    nn.Linear(
        in_features = 32*5*5,
        out_features = 10
    )

)

In [10]:
learning_rate = 0.0001
epochs = 20

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

In [12]:
from torch.utils.data import DataLoader

def train(model, train_loader, optimizer, loss_fn, epochs):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if batch_idx % 50 == 49:
                print(f'Epoch [{epoch + 1}/{epochs}], Batch [{batch_idx + 1}/{len(train_loader)}], Loss: {running_loss / 50:.4f}')
                running_loss = 0.0

        avg_epoch_loss = running_loss / (len(train_loader) % 50 if len(train_loader) % 50 != 0 else 50) if running_loss > 0 else 0.0
        print(f'End of Epoch {epoch + 1}, Average Loss: {avg_epoch_loss:.4f}')

    print('Finished Training')
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)
train(model, train_loader, optimizer, loss_fn, epochs)

Epoch [1/20], Batch [50/938], Loss: 0.0450
Epoch [1/20], Batch [100/938], Loss: 0.0506
Epoch [1/20], Batch [150/938], Loss: 0.0448
Epoch [1/20], Batch [200/938], Loss: 0.0483
Epoch [1/20], Batch [250/938], Loss: 0.0492
Epoch [1/20], Batch [300/938], Loss: 0.0426
Epoch [1/20], Batch [350/938], Loss: 0.0412
Epoch [1/20], Batch [400/938], Loss: 0.0504
Epoch [1/20], Batch [450/938], Loss: 0.0459
Epoch [1/20], Batch [500/938], Loss: 0.0453
Epoch [1/20], Batch [550/938], Loss: 0.0533
Epoch [1/20], Batch [600/938], Loss: 0.0514
Epoch [1/20], Batch [650/938], Loss: 0.0396
Epoch [1/20], Batch [700/938], Loss: 0.0469
Epoch [1/20], Batch [750/938], Loss: 0.0509
Epoch [1/20], Batch [800/938], Loss: 0.0472
Epoch [1/20], Batch [850/938], Loss: 0.0492
Epoch [1/20], Batch [900/938], Loss: 0.0385
End of Epoch 1, Average Loss: 0.0367
Epoch [2/20], Batch [50/938], Loss: 0.0414
Epoch [2/20], Batch [100/938], Loss: 0.0410
Epoch [2/20], Batch [150/938], Loss: 0.0452
Epoch [2/20], Batch [200/938], Loss: 0.06

In [14]:
def test(model, test_loader, loss_fn):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            test_loss += loss_fn(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({100. * correct / len(test_loader.dataset):.0f}%)\n')
test(model, test_loader, loss_fn)


Test set: Average loss: 0.0000, Accuracy: 9874/10000 (99%)



In [17]:
def train_accuracy(model, train_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in train_loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    print(f'\nTrain set: Accuracy: {correct}/{len(train_loader.dataset)} ({100. * correct / len(train_loader.dataset):.0f}%)\n')


In [18]:
print('Evaluating Training Accuracy:')
train_accuracy(model, train_loader)

print('Evaluating Test Accuracy:')
test(model, test_loader, loss_fn)


Evaluating Training Accuracy:

Train set: Accuracy: 59544/60000 (99%)

Evaluating Test Accuracy:

Test set: Average loss: 0.0000, Accuracy: 9874/10000 (99%)



In [21]:

test(model, test_loader, loss_fn)


Test set: Average loss: 0.0000, Accuracy: 9874/10000 (99%)



## Exporting the Weights

In [23]:
conv2_layer = model[3]
conv2_weights = conv2_layer.weight.cpu().detach().numpy()
output_filename = 'conv2_weights.txt'
np.savetxt(output_filename, conv2_weights.reshape(-1, conv2_weights.shape[-1]))

print(f"Weights of the second convolutional layer exported to '{output_filename}'")

Weights of the second convolutional layer exported to 'conv2_weights.txt'
